# 完整流程示例（对照 CLI 页）

等高层要素在 `resource/data/`，降水在 `resource/EC/APCP01/`，实况在 `resource/CMPAS_APCP/`。批量调度按目录列出 nc，插值用 `interp_gg_linear`，中间文件用 `meb.write_griddata_to_nc`。

**本窗口**

- 个例：2026-08-04–06 起报 `00`，时效 `003–021`。
- 跳过残缺时次：`2026080500.018`（整时次缺要素）、`2026080500.012`（缺 `UGRD/950`）。
- 订正方法为原流程 `ApplyOrographicEnhancement`（先减模式增强，再加高分辨率增强）。
- 检验表写到 `output/verify/hourly_20260804_06.csv`。

**调度**

- 模式增强：同一套等高层要素配 0.2° 模式地形。得到的增强场插到 0.1° 降水网格上，`subtract`。
- 高分辨率增强：同一套要素配 1 km 地形。扣完的降水再插到这张地形上，`add`。
- 中间文件都在 `output/_tmp/`：`oe_model_native.nc`、`oe_model.nc`、`oe_1km.nc`、`precip.nc`，每个时次覆盖上一时次。
- 时效 `000` 没有 `APCP01`，不进入订正。

**时间与量**

- EC 起报与 `dtime` 是世界时。有效世界时 = 起报 + 时效。
- `APCP01` 每 3 h 一个文件，数值是以有效时为终点的 **3 小时累计平均**（按 `mm h-1` 使用，不再除以 3）。
- CMPAS 文件名和文件内 `time` 都是北京时，逐小时。北京时 = 有效世界时 + 8 h。
- 检验取北京时终点及其前两个整点，三小时平均后与预报比较。


## 1. 准备


In [ ]:
from __future__ import annotations

import sys
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import meteva_base as meb

PKG = Path("..").resolve()
sys.path[:0] = [str(PKG.parent), str(PKG)]

from orographic_precipitation_downscaling.cli.orographic_enhancement_cli import process as oe_process
from orographic_precipitation_downscaling.cli.apply_orographic_enhancement_cli import process as apply_oe

DATA = PKG / "resource" / "data"
OBS_ROOT = PKG / "resource" / "CMPAS_APCP"
OUT = DATA / "output"

OROG_MODEL = str(PKG / "resource" / "ancil" / "EC_D1D_orography_20km.nc")
OROG = str(PKG / "resource" / "ancil" / "HGT_M_1KM_CHINA.nc")
FIG = OUT / "figs"
PRECIP_TMP = str(OUT / "_tmp" / "precip.nc")
OE_MODEL_NATIVE = str(OUT / "_tmp" / "oe_model_native.nc")
OE_MODEL_TMP = str(OUT / "_tmp" / "oe_model.nc")
OE_1KM_TMP = str(OUT / "_tmp" / "oe_1km.nc")

# 8 月 4–6 日窗口；残缺时次不进入订正
CASE_INITS = ("2026080400", "2026080500", "2026080600")
SKIP_TAGS = {"2026080500.012", "2026080500.018"}


def nc_index(root: Path) -> dict:
    """递归列出目录下的 nc，以文件名主干为键。"""
    files = meb.path_tools.get_path_list_in_dir(str(root))
    return {Path(p).stem: p for p in files if str(p).endswith(".nc")}


tmp_ix = nc_index(DATA / "TMP")
rh_ix = nc_index(DATA / "RH")
pres_ix = nc_index(DATA / "PRES")
u_ix = nc_index(DATA / "UGRD")
v_ix = nc_index(DATA / "VGRD")
apcp_list = meb.path_tools.get_path_list_in_dir(str(PKG / "resource" / "EC" / "APCP01"))
apcp_list = sorted(
    p
    for p in apcp_list
    if str(p).endswith(".nc")
    and Path(p).stem in tmp_ix
    and Path(p).stem.split(".", 1)[0] in CASE_INITS
    and Path(p).stem not in SKIP_TAGS
)


def parse_tag(tag: str):
    init_s, lead_s = tag.split(".")
    return datetime.strptime(init_s, "%Y%m%d%H"), int(lead_s)


print("n forecasts", len(apcp_list))
print("example APCP", apcp_list[0] if apcp_list else None)
print("precip tmp", PRECIP_TMP)


def cmpas_path(bjt: datetime) -> Path:
    """北京时整点对应的逐小时实况路径。"""
    return OBS_ROOT / f"{bjt:%Y}" / f"{bjt:%Y%m%d}" / f"BJT_{bjt:%y%m%d%H}.000.nc"


def obs_3h_mean(valid_utc: datetime, grid):
    """以有效世界时为终点，取北京时三小时实况平均，并落到目标网格。"""
    bjt_end = valid_utc + timedelta(hours=8)
    hours = [bjt_end - timedelta(hours=2), bjt_end - timedelta(hours=1), bjt_end]
    acc = None
    last = None
    for bjt in hours:
        path = cmpas_path(bjt)
        if not path.exists():
            raise FileNotFoundError(path)
        da = meb.read_griddata_from_nc(str(path))
        da = meb.interp_gg_linear(da, grid, outer_value=np.nan)
        values = np.asarray(da.values, dtype=np.float64)
        acc = values if acc is None else acc + values
        last = da
    out = last.copy(data=(acc / 3.0).astype(np.float32))
    out.attrs["units"] = "mm h-1"
    out.name = "cmpas_3h_mean"
    return out


## 2. 批量估增强 → 降水插到 1 km → Apply

原订正：`subtract` 模式增强 → 插到 1 km → `add` 高分辨率增强。

`RUN_APPLY` 为 `False` 且该时次订正结果已存在时跳过。两套增强场只写 `output/_tmp/`，下一时次覆盖。模式增强是 `m s-1`，用 xarray 直写。降水中间文件用 `meb.write_griddata_to_nc`。


In [ ]:
RUN_APPLY = True

dest = meb.read_griddata_from_nc(OROG)
dest.attrs["units"] = "m"
grid1km = meb.get_grid_of_data(dest)

for apcp in apcp_list:
    tag = Path(apcp).stem
    corr_out = str(OUT / "corr" / f"{tag}.nc")
    if not RUN_APPLY and Path(corr_out).exists():
        continue

    oe_process(
        tmp_ix[tag], rh_ix[tag], pres_ix[tag], u_ix[tag], v_ix[tag],
        OROG_MODEL,
        output_path=OE_MODEL_NATIVE,
    )
    oe_process(
        tmp_ix[tag], rh_ix[tag], pres_ix[tag], u_ix[tag], v_ix[tag],
        OROG,
        output_path=OE_1KM_TMP,
    )

    precip = meb.read_griddata_from_nc(apcp)
    precip.attrs["units"] = "mm h-1"
    meb.write_griddata_to_nc(precip, PRECIP_TMP, creat_dir=True)
    oe_model = meb.read_griddata_from_nc(OE_MODEL_NATIVE)
    oe_on_precip = meb.interp_gg_linear(
        oe_model, meb.get_grid_of_data(precip), outer_value=np.nan
    )
    oe_on_precip.attrs["units"] = oe_model.attrs.get("units") or "m s-1"
    oe_name = oe_on_precip.name or "orographic_enhancement"
    Path(OE_MODEL_TMP).parent.mkdir(parents=True, exist_ok=True)
    oe_on_precip.to_dataset(name=oe_name).to_netcdf(OE_MODEL_TMP)
    residual = apply_oe(
        PRECIP_TMP, OE_MODEL_TMP, output_path=None, operation="subtract"
    )
    fine = meb.interp_gg_linear(residual, grid1km, outer_value=np.nan)
    fine.attrs["units"] = "mm h-1"
    fine.name = "precipitation_rate"
    meb.write_griddata_to_nc(fine, PRECIP_TMP, creat_dir=True)
    apply_oe(PRECIP_TMP, OE_1KM_TMP, output_path=corr_out, operation="add")
    print("corr", tag)

print("apply done")


## 3. 三张图（抽第 1 个时次）

1 订正前（mm/h）　2 订正后（mm/h）　3 实况 3 小时平均（mm/h）  
三张图共用同一套色标。细网格绘图抽稀。


In [ ]:
from meteva_base.tool.plot_tools import pcolormesh_2d_grid

apcp = apcp_list[0]
tag = Path(apcp).stem
init, lead = parse_tag(tag)
valid = init + timedelta(hours=lead)
step = 10

precip = meb.read_griddata_from_nc(apcp)
precip.attrs["units"] = "mm h-1"
before = meb.interp_gg_linear(precip, grid1km, outer_value=np.nan)
before.attrs["units"] = "mm h-1"
before.name = "precipitation_rate"
after = meb.read_griddata_from_nc(str(OUT / "corr" / f"{tag}.nc"))
obs = obs_3h_mean(valid, grid1km)

before = before.isel(lat=slice(None, None, step), lon=slice(None, None, step))
after = after.isel(lat=slice(None, None, step), lon=slice(None, None, step))
obs = obs.isel(lat=slice(None, None, step), lon=slice(None, None, step))

both = np.concatenate([
    np.asarray(before.values, dtype=float).ravel(),
    np.asarray(after.values, dtype=float).ravel(),
    np.asarray(obs.values, dtype=float).ravel(),
])
both = both[np.isfinite(both) & (both >= 0) & (both < 1e10)]
hi = max(float(np.nanpercentile(both, 99)), 1.0)
clevs = [0, 0.1, 1, 2, 4, 8, 16, 32]
while clevs[-1] < hi and len(clevs) < 19:
    clevs.append(clevs[-1] * 2)

bjt = valid + timedelta(hours=8)
pcolormesh_2d_grid(
    before, save_path=str(FIG / f"{tag}_1_before.png"),
    title=f"1 订正前（mm/h） {tag}",
    clevs=clevs, cmap="rain_1h", show=True,
)
pcolormesh_2d_grid(
    after, save_path=str(FIG / f"{tag}_2_after.png"),
    title=f"2 订正后（mm/h） {tag}",
    clevs=clevs, cmap="rain_1h", show=True,
)
pcolormesh_2d_grid(
    obs, save_path=str(FIG / f"{tag}_3_obs.png"),
    title=f"3 实况 3h 平均（mm/h） BJT {bjt-timedelta(hours=2):%H}-{bjt:%Y%m%d%H}",
    clevs=clevs, cmap="rain_1h", show=True,
)
print("figs:", FIG)


## 4. 验证 Bias / RMSE（订正前 / 订正后 vs 实况）

- **订正前**：原始降水插到 1 km，不含增强项。
- **订正后**：先在 0.1° 上扣除 0.2° 模式增强，再在 1 km 上叠加高分辨率增强（原 Apply 流程）。
- **实况**：有效世界时对应的北京时终点及前两个整点，三小时平均后再插到 1 km。
- 同一掩码（有限值交集）。ΔRMSE = 订正后 − 订正前，负值表示改善。
- 结果表：`output/verify/hourly_20260804_06.csv`。


In [ ]:
def _bias_rmse(pred: np.ndarray, obs: np.ndarray) -> tuple[float, float, int]:
    mask = np.isfinite(pred) & np.isfinite(obs) & (np.abs(pred) < 1e10) & (np.abs(obs) < 1e10)
    if not np.any(mask):
        return float("nan"), float("nan"), 0
    diff = pred[mask] - obs[mask]
    return float(np.mean(diff)), float(np.sqrt(np.mean(diff**2))), int(mask.sum())


rows = []
for apcp in apcp_list:
    tag = Path(apcp).stem
    init, lead = parse_tag(tag)
    valid = init + timedelta(hours=lead)
    precip = meb.read_griddata_from_nc(apcp)
    precip.attrs["units"] = "mm h-1"
    before = meb.interp_gg_linear(precip, grid1km, outer_value=np.nan)
    before.attrs["units"] = "mm h-1"
    before.name = "precipitation_rate"
    after = meb.read_griddata_from_nc(str(OUT / "corr" / f"{tag}.nc"))
    obs = obs_3h_mean(valid, grid1km)
    b = np.asarray(obs.values, dtype=float).ravel()
    a0 = np.asarray(before.values, dtype=float).ravel()
    a1 = np.asarray(after.values, dtype=float).ravel()
    bias0, rmse0, n = _bias_rmse(a0, b)
    bias1, rmse1, _ = _bias_rmse(a1, b)
    rows.append({
        "tag": tag,
        "valid_utc": valid.strftime("%Y%m%d%H"),
        "bjt_end": (valid + timedelta(hours=8)).strftime("%Y%m%d%H"),
        "bias_before": bias0,
        "rmse_before": rmse0,
        "bias_after": bias1,
        "rmse_after": rmse1,
        "delta_rmse": rmse1 - rmse0,
        "n": n,
    })
    print(
        f"{tag}  before bias={bias0:.3f} rmse={rmse0:.3f} | "
        f"after bias={bias1:.3f} rmse={rmse1:.3f} | "
        f"Δrmse={rmse1 - rmse0:+.3f} mm/h"
    )

hourly = pd.DataFrame(rows)
ver_dir = OUT / "verify"
ver_dir.mkdir(parents=True, exist_ok=True)
hourly_path = ver_dir / "hourly_20260804_06.csv"
hourly.to_csv(hourly_path, index=False)
print(
    f"mean  before bias={hourly.bias_before.mean():.3f} rmse={hourly.rmse_before.mean():.3f} | "
    f"after bias={hourly.bias_after.mean():.3f} rmse={hourly.rmse_after.mean():.3f} | "
    f"Δrmse={hourly.delta_rmse.mean():+.3f} mm/h"
)
print("table", hourly_path)
